In [22]:
import numpy as np
import pandas as pd

In [23]:
oa_dict = pd.read_csv('OA_Lexicon_eBL.csv')
oa_dict

,type,form,norm,lexeme,eBL,I_IV,A_D,Female(f),Alt_lex
0,word,áb ša-ra-ni,ab šarrānē,ab šarrānē,https://www.ebl.lmu.de/dictionary?word=ab šarrānē,NaN,NaN,NaN,NaN
1,word,áb ša-ra-nim,ab šarrānem,ab šarrānē,https://www.ebl.lmu.de/dictionary?word=ab šarrānē,NaN,NaN,NaN,NaN
2,word,áb-ša-ra-nim,ab šarrānem,ab šarrānē,https://www.ebl.lmu.de/dictionary?word=ab šarrānē,NaN,NaN,NaN,NaN
3,word,áb-ša-ra-ni,ab šarrānē,ab šarrānē,https://www.ebl.lmu.de/dictionary?word=ab šarrānē,NaN,NaN,NaN,NaN
4,word,áb-ša-ra-ni₇,ab šarrānē,ab šarrānē,https://www.ebl.lmu.de/dictionary?word=ab šarrānē,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
39327,PN,zu-zu-ta-ì-lí,Zuzuta-ilī,Zuzuta-ilī,https://www.ebl.lmu.de/dictionary?word=Zuzuta-ilī,NaN,NaN,NaN,NaN
39328,PN,zu-zu-tí,Zuzuti,Zuzuti,https://www.ebl.lmu.de/dictionary?word=Zuzuti,NaN,NaN,NaN,NaN
39329,PN,zu-zu-za-num,Zuzuzānum,Zuzuzānu,https://www.ebl.lmu.de/dictionary?word=Zuzuzānu,NaN,NaN,NaN,NaN
39330,PN,zu-zu-za-nim,Zuzuzānim,Zuzuzānu,https://www.ebl.lmu.de/dictionary?word=Zuzuzānu,NaN,NaN,NaN,NaN


In [24]:
train = pd.read_csv('train.csv')
train

,oare_id,transliteration,translation
0,004a7dbd-57ce-46f8-9691-409be61c676e,KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-(d)IM KIŠI...,"Seal of Mannum-balum-Aššur son of Ṣilli-Adad, ..."
1,0064939c-59b9-4448-a63d-34612af0a1b5,1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé,Itūr-ilī has received one textile of ordinary ...
2,0073f2c0-524c-4bbf-915a-8c1772a4fb98,TÚG u-la i-dí-na-ku-um i-tù-ra-ma 9 GÍN KÙ.BABBAR,... he did not give you a textile. He returned...
3,009fb838-8038-42bc-ad34-5f795b3840ee,KIŠIB šu-(d)EN.LÍL DUMU šu-ku-bi-im KIŠIB ṣí-l...,"Seal of Šu-Illil son of Šu-Kūbum, seal of Ṣilū..."
4,00aa1c55-c80c-4346-a159-73ad43ab0ff7,um-ma šu-ku-tum-ma a-na IŠTAR-lá-ma-sí ù ni-ta...,From Šukkutum to Ištar-lamassī and Nitahšušar:...
...,...,...,...
1556,ff3208e4-8ab8-4368-b4df-7b80afa5bc32,um-ma en-nam-a-šur-ma a-na a-la-ḫi-im qí-bi-ma...,From Ennam-Aššur to Ali-ahum: Here 2 men have ...
1557,ff43a284-3d67-4238-8b4a-9b6cb7531e0a,3 ma-na KÙ.BABBAR ṣa-ru-pá-am i-na ší-im SÍG.Ḫ...,Ilī-ašrannī son of Sukkalliya has received 3 m...
1558,ff5747a4-af8a-4100-a906-a2660ae72606,ša-lim-a-šùr a-na a-mur-IŠTAR ú-ṭá-ḫi-ni-a-tí-...,Šalim-Aššur made us approach Amur-Ištar and Ša...
1559,ff777871-97ce-4bfc-bdfb-73352868944d,a-na en-nam-a-šùr qí-bi-ma um-ma IŠTAR-ra-bi₄-...,To Ennam-Aššur from Ištar-rabiʾat: With respec...


In [18]:
import pandas as pd
import re
from typing import List, Dict, Any

def pos_tag_akkadian(df_train: pd.DataFrame, df_dict: pd.DataFrame, text_col: str = 'transliteration', 
                     word_col: str = 'form', type_col: str = 'type') -> pd.DataFrame:
    """
    Performs POS tagging on Akkadian transliteration paragraphs in train DataFrame
    using the dictionary DataFrame. Adds a 'pos_tags' column with tagged tokens.
    
    Assumes:
    - Text split by spaces into words (handles hyphenated syllables as single tokens).
    - Dictionary matching is case-sensitive exact match.
    - Unknown words tagged as 'UNK'.
    
    Args:
        df_train: DataFrame with 'transliteration' column containing paragraphs.
        df_dict: DataFrame with 'word' and 'type' columns.
        text_col: Name of text column in train.
        word_col: Name of word column in dict.
        type_col: Name of POS type column in dict.
    
    Returns:
        Copy of df_train with new 'pos_tags' column containing list of (token, tag) tuples.
    """
    word_to_pos = dict(zip(df_dict[word_col], df_dict[type_col]))
    
    def tag_paragraph(text: str) -> List[tuple[str, str]]:
        if pd.isna(text):
            return []
        words = re.split(r'\s+', str(text).strip())
        tagged = []
        for word in words:
            clean_word = re.sub(r'[^\w-]', '', word.strip())  
            tag = word_to_pos.get(clean_word, 'UNK')
            tagged.append((word, tag))  
        return tagged
    
    df_train = df_train.copy()
    df_train['pos_tags'] = df_train[text_col].apply(tag_paragraph)
    
    return df_train
train_tagged = pos_tag_akkadian(train, oa_dict)
train_tagged

,oare_id,transliteration,translation,pos_tags
0,004a7dbd-57ce-46f8-9691-409be61c676e,KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-(d)IM KIŠI...,"Seal of Mannum-balum-Aššur son of Ṣilli-Adad, ...","[(KIŠIB, word), (ma-nu-ba-lúm-a-šur, PN), (DUM..."
1,0064939c-59b9-4448-a63d-34612af0a1b5,1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé,Itūr-ilī has received one textile of ordinary ...,"[(1, UNK), (TÚG, word), (ša, word), (qá-tim, w..."
2,0073f2c0-524c-4bbf-915a-8c1772a4fb98,TÚG u-la i-dí-na-ku-um i-tù-ra-ma 9 GÍN KÙ.BABBAR,... he did not give you a textile. He returned...,"[(TÚG, word), (u-la, word), (i-dí-na-ku-um, wo..."
3,009fb838-8038-42bc-ad34-5f795b3840ee,KIŠIB šu-(d)EN.LÍL DUMU šu-ku-bi-im KIŠIB ṣí-l...,"Seal of Šu-Illil son of Šu-Kūbum, seal of Ṣilū...","[(KIŠIB, word), (šu-(d)EN.LÍL, UNK), (DUMU, wo..."
4,00aa1c55-c80c-4346-a159-73ad43ab0ff7,um-ma šu-ku-tum-ma a-na IŠTAR-lá-ma-sí ù ni-ta...,From Šukkutum to Ištar-lamassī and Nitahšušar:...,"[(um-ma, word), (šu-ku-tum-ma, PN), (a-na, wor..."
...,...,...,...,...
1556,ff3208e4-8ab8-4368-b4df-7b80afa5bc32,um-ma en-nam-a-šur-ma a-na a-la-ḫi-im qí-bi-ma...,From Ennam-Aššur to Ali-ahum: Here 2 men have ...,"[(um-ma, word), (en-nam-a-šur-ma, PN), (a-na, ..."
1557,ff43a284-3d67-4238-8b4a-9b6cb7531e0a,3 ma-na KÙ.BABBAR ṣa-ru-pá-am i-na ší-im SÍG.Ḫ...,Ilī-ašrannī son of Sukkalliya has received 3 m...,"[(3, UNK), (ma-na, word), (KÙ.BABBAR, UNK), (ṣ..."
1558,ff5747a4-af8a-4100-a906-a2660ae72606,ša-lim-a-šùr a-na a-mur-IŠTAR ú-ṭá-ḫi-ni-a-tí-...,Šalim-Aššur made us approach Amur-Ištar and Ša...,"[(ša-lim-a-šùr, PN), (a-na, word), (a-mur-IŠTA..."
1559,ff777871-97ce-4bfc-bdfb-73352868944d,a-na en-nam-a-šùr qí-bi-ma um-ma IŠTAR-ra-bi₄-...,To Ennam-Aššur from Ištar-rabiʾat: With respec...,"[(a-na, word), (en-nam-a-šùr, PN), (qí-bi-ma, ..."


In [21]:
oa_dict[oa_dict['form'] == 'i-na']

,type,form,norm,lexeme,eBL,I_IV,A_D,Female(f),Alt_lex
13745,word,i-na,ina,ina,https://www.ebl.lmu.de/dictionary?word=ina,NaN,NaN,NaN,NaN
13796,PN,i-na,Innā,Innā,https://www.ebl.lmu.de/dictionary?word=Innā,NaN,NaN,NaN,NaN


In [25]:
import pandas as pd
import re
from typing import List, Dict, Any

def normalize_and_pos_tag_akkadian(df_train: pd.DataFrame, df_dict: pd.DataFrame, 
                                  text_col: str = 'transliteration',
                                  word_col: str = 'form', norm_col: str = 'norm', 
                                  type_col: str = 'type') -> pd.DataFrame:
    """
    1. Normalizes Akkadian transliteration using 'norm' column from oa_dict.
    2. Applies POS tagging using 'type' column.
    
    Normalization: Replaces known words with their normalized form from oa_dict.
    POS Tagging: Tags normalized tokens with types (word/PN/GN) or UNK.
    PN: Person Name
    GN: Geographical Name
    word: normal word
    UNK: unknown(not in oa_lexicon_dictionary)
    Args:
        df_train: DataFrame with 'transliteration' column (paragraphs).
        df_dict: DataFrame with 'word', 'norm', 'type' columns.
        text_col/word_col/norm_col/type_col: Column names.
    
    Returns:
        Copy of df_train with 'normalized_text' (str), 'norm_tokens' (list), 
        'pos_tags' (list of (norm_token, tag)) columns.
    """
    df_train = df_train.copy()
    
    word_to_norm = dict(zip(df_dict[word_col], df_dict[norm_col]))
    word_to_pos = dict(zip(df_dict[word_col], df_dict[type_col]))
    
    def process_paragraph(text: str) -> Dict[str, Any]:
        if pd.isna(text):
            return {'normalized_text': '', 'norm_tokens': [], 'pos_tags': []}
        
        original_words = re.split(r'\s+', str(text).strip())
        normalized_words = []
        pos_tags = []
        
        for orig_word in original_words:
            clean_word = re.sub(r'[^\wšṣṭḫḥŋ-]', '', orig_word.strip())
            
            norm_form = word_to_norm.get(clean_word, clean_word)
            normalized_words.append(norm_form)
            
            tag = word_to_pos.get(clean_word, 'UNK')
            pos_tags.append((norm_form, tag))
        
        normalized_text = ' '.join(normalized_words)
        
        return {
            'normalized_text': normalized_text,
            'norm_tokens': normalized_words,
            'pos_tags': pos_tags
        }
    
    results = df_train[text_col].apply(process_paragraph)
    df_train['normalized_text'] = results.apply(lambda x: x['normalized_text'])
    df_train['norm_tokens'] = results.apply(lambda x: x['norm_tokens'])
    df_train['pos_tags'] = results.apply(lambda x: x['pos_tags'])
    
    return df_train

train_processed = normalize_and_pos_tag_akkadian(train, oa_dict)
train_processed

,oare_id,transliteration,translation,normalized_text,norm_tokens,pos_tags
0,004a7dbd-57ce-46f8-9691-409be61c676e,KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-(d)IM KIŠI...,"Seal of Mannum-balum-Aššur son of Ṣilli-Adad, ...",kunukkum Mannum-bālum-Aššur mer'ēn ṣí-lá-dIM k...,"[kunukkum, Mannum-bālum-Aššur, mer'ēn, ṣí-lá-d...","[(kunukkum, word), (Mannum-bālum-Aššur, PN), (..."
1,0064939c-59b9-4448-a63d-34612af0a1b5,1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé,Itūr-ilī has received one textile of ordinary ...,1 TÚG ša qātem Itūr-ilī ilqe,"[1, TÚG, ša, qātem, Itūr-ilī, ilqe]","[(1, UNK), (TÚG, word), (ša, word), (qātem, wo..."
2,0073f2c0-524c-4bbf-915a-8c1772a4fb98,TÚG u-la i-dí-na-ku-um i-tù-ra-ma 9 GÍN KÙ.BABBAR,... he did not give you a textile. He returned...,TÚG ula iddinākum itūramma 9 šiqlam KÙBABBAR,"[TÚG, ula, iddinākum, itūramma, 9, šiqlam, KÙB...","[(TÚG, word), (ula, word), (iddinākum, word), ..."
3,009fb838-8038-42bc-ad34-5f795b3840ee,KIŠIB šu-(d)EN.LÍL DUMU šu-ku-bi-im KIŠIB ṣí-l...,"Seal of Šu-Illil son of Šu-Kūbum, seal of Ṣilū...",kunukkum šu-dENLÍL mer'ēn Šu-Kūbim kunukkum Ṣi...,"[kunukkum, šu-dENLÍL, mer'ēn, Šu-Kūbim, kunukk...","[(kunukkum, word), (šu-dENLÍL, UNK), (mer'ēn, ..."
4,00aa1c55-c80c-4346-a159-73ad43ab0ff7,um-ma šu-ku-tum-ma a-na IŠTAR-lá-ma-sí ù ni-ta...,From Šukkutum to Ištar-lamassī and Nitahšušar:...,umma Šukkutumma anna Ištar-lamassī u Nitaḫšuša...,"[umma, Šukkutumma, anna, Ištar-lamassī, u, Nit...","[(umma, word), (Šukkutumma, PN), (anna, word),..."
...,...,...,...,...,...,...
1556,ff3208e4-8ab8-4368-b4df-7b80afa5bc32,um-ma en-nam-a-šur-ma a-na a-la-ḫi-im qí-bi-ma...,From Ennam-Aššur to Ali-ahum: Here 2 men have ...,umma Ennam-Aššur-ma anna Al-aḫim qibima annāka...,"[umma, Ennam-Aššur-ma, anna, Al-aḫim, qibima, ...","[(umma, word), (Ennam-Aššur-ma, PN), (anna, wo..."
1557,ff43a284-3d67-4238-8b4a-9b6cb7531e0a,3 ma-na KÙ.BABBAR ṣa-ru-pá-am i-na ší-im SÍG.Ḫ...,Ilī-ašrannī son of Sukkalliya has received 3 m...,3 manā KÙBABBAR ṣarrupam Innā ši'im SÍGḪIA ša ...,"[3, manā, KÙBABBAR, ṣarrupam, Innā, ši'im, SÍG...","[(3, UNK), (manā, word), (KÙBABBAR, UNK), (ṣar..."
1558,ff5747a4-af8a-4100-a906-a2660ae72606,ša-lim-a-šùr a-na a-mur-IŠTAR ú-ṭá-ḫi-ni-a-tí-...,Šalim-Aššur made us approach Amur-Ištar and Ša...,Šalim-Aššur anna Amur-Ištar uṭaḫḫiniātima umma...,"[Šalim-Aššur, anna, Amur-Ištar, uṭaḫḫiniātima,...","[(Šalim-Aššur, PN), (anna, word), (Amur-Ištar,..."
1559,ff777871-97ce-4bfc-bdfb-73352868944d,a-na en-nam-a-šùr qí-bi-ma um-ma IŠTAR-ra-bi₄-...,To Ennam-Aššur from Ištar-rabiʾat: With respec...,anna Ennam-Aššur qibima umma Ištar-rabiat-ma a...,"[anna, Ennam-Aššur, qibima, umma, Ištar-rabiat...","[(anna, word), (Ennam-Aššur, PN), (qibima, wor..."


In [27]:
train_processed['pos_tags'][0]

[('kunukkum', 'word'),
 ('Mannum-bālum-Aššur', 'PN'),
 ("mer'ēn", 'word'),
 ('ṣí-lá-dIM', 'UNK'),
 ('kunukkum', 'word'),
 ('šu-dENLÍL', 'UNK'),
 ("mer'ēn", 'word'),
 ('Mannum-kī-Aššur', 'PN'),
 ('kunukkum', 'word'),
 ('Puzur-Aššur', 'PN'),
 ("mer'ēn", 'word'),
 ('Adā', 'PN'),
 ('033333', 'UNK'),
 ('manā', 'word'),
 ('2', 'UNK'),
 ('šiqlam', 'word'),
 ('KÙBABBAR', 'UNK'),
 ('damqān', 'word'),
 ('iṣṣēr', 'word'),
 ('Puzur-Aššur', 'PN'),
 ("mer'ēn", 'word'),
 ('Adā', 'PN'),
 ('Al-aḫum', 'PN'),
 ('īšu', 'word'),
 ('ištu', 'word'),
 ('ḫamuštem', 'word'),
 ('ša', 'word'),
 ('Ilī-dān', 'PN'),
 ('ITUKAM', 'UNK'),
 ('ša', 'word'),
 ('kēnātem', 'word'),
 ('līmum', 'word'),
 ('Enna-Suen', 'PN'),
 ('anna', 'word'),
 ('warḫam', 'word'),
 ('14', 'UNK'),
 ('ḫamšātem', 'word'),
 ('išaqqal', 'word'),
 ('šumma', 'word'),
 ('lā', 'word'),
 ('išqul', 'word'),
 ('15', 'UNK'),
 ('GÍNTA', 'UNK'),
 ('anna', 'word'),
 ('1', 'UNK'),
 ("manā'em", 'word'),
 ('Innā', 'PN'),
 ('ITU1KAM', 'UNK'),
 ('ṣibtam', 'word')

In [38]:
train_processed.to_csv('pos_train.csv')

In [33]:
oa_dict[oa_dict['form'] == 'KIŠIB']

,type,form,norm,lexeme,eBL,I_IV,A_D,Female(f),Alt_lex
16738,word,KIŠIB,kunuk,kunukku,https://www.ebl.lmu.de/dictionary?word=kunukku,NaN,NaN,NaN,NaN
16773,word,KIŠIB,kunukkum,kunukku,https://www.ebl.lmu.de/dictionary?word=kunukku,NaN,NaN,NaN,NaN


In [37]:
oa_dict['form'].duplicated().sum()

4283